In [ ]:
import pandas as pd
import numpy as np
import time
from collections import Counter
from sklearn.metrics import confusion_matrix

In [ ]:
class MyStandardScaler:
    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
    
    def transform(self, X):
        try:
            X = (X - self.mean)/self.std
        except Exception:
            print("You have to fit the data first")
        return X

    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [ ]:
def train_test_split(x, y, seed=10, testsize=0.5):
    np.random.seed(seed)

    indices = np.random.permutation(len(x))

    x, y = x[indices], y[indices]

    index = int(len(y)*testsize) 
    
    return  np.array(x[index:]), np.array(x[:index]), np.array(y[index:]), np.array(y[:index]) # Initialise the variable in the order: X_train, X_test, Y_train, Y_test

def accuracy_score(y_test, y_pred):
    correct = 0
    for test, pred in zip(y_test, y_pred):
        if test == pred:
            correct += 1
    
    return correct/len(y_test)

In [ ]:

def KNNClassifier(X_train, X_test, y_train, k):
    bef_train = time.time()
    dist = [np.sqrt(np.sum((X_train - X_test[n])**2, axis=1)) for n in range(len(X_test))]
    sorted_dist = np.argsort(np.array(dist))
    closest = [y_train[n] for n in sorted_dist[:, :k]]
    aft_train = time.time()
    training_time = aft_train-bef_train
    
    bef_test = time.time()
    predictions = []
    for arr in closest:
        values, counts = np.unique(arr, return_counts=True)
        modal = values[np.argmax(counts)]
        predictions.append(modal)
    aft_test = time.time()
    testing_time = aft_test-bef_test

    return predictions, training_time, testing_time

In [ ]:
def build_tree(X_train, y_train, depth=0, max_depth=3):

    current_depth = depth
    if current_depth >= max_depth:
        return {'leaf' : True, 'prediction' : Counter(y_train).most_common(1)[0][0]}
    
    best_gini = float('inf')
    best_feature = None
    best_threshold = None
    best_x_left = None
    best_x_right = None
    best_y_left = None
    best_y_right = None


    for feature in range(X_train.shape[1]):
        for threshold in np.unique(X_train[:, feature]):
            
            x_left = []
            x_right = []
            y_left = []
            y_right = []

            for i in range(len(X_train)):
                if X_train[i][feature] <= threshold:
                    x_left.append(X_train[i])
                    y_left.append(y_train[i])
                else:
                    x_right.append(X_train[i])
                    y_right.append(y_train[i])

            if len(y_left) == 0 or len(y_right) == 0:
                continue
            #calculate gini left
            unique_left, counts_left = np.unique(y_left, return_counts=True)


            p_left = counts_left / len(y_left)
            gini_left = 1 - np.sum(p_left**2)
            #calculate gini right
            unique_right, counts_right = np.unique(y_right, return_counts=True)

            p_right = counts_right / len(y_right)
            gini_right = 1 - np.sum(p_right**2)

            #weighted gini

            gini = (len(y_left)*gini_left + len(y_right)*gini_right) / len(y_train)

            if gini < best_gini:
                best_gini = gini
                best_feature = feature
                best_threshold = threshold
                best_x_left = x_left
                best_x_right = x_right
                best_y_left = y_left
                best_y_right = y_right
    if best_feature is None:
        return {'leaf' : True, 'prediction' : Counter(y_train).most_common(1)[0][0]}

    current_depth += 1

    node = {
        'feature' : best_feature,
        'threshold' : best_threshold,
        'left' : build_tree(np.array(best_x_left), np.array(best_y_left), depth=current_depth, max_depth=3),
        'right' : build_tree(np.array(best_x_right), np.array(best_y_right), depth=current_depth, max_depth=3)
    }

    return node

In [ ]:
def DTClassifier(tree, X_test):
    predictions = []

    for sample in X_test:

        node = tree
        while 'prediction' not in node:
            if sample[node['feature']] <= node['threshold']:
                node = node['left']
            else:
                node = node['right']
        
        predictions.append(node['prediction'])
    
    return predictions

In [ ]:
def probability(data, value):
    return np.count_nonzero(data==value)/len(data)


def NBClassifier(X_test, X_train, y_train):
    means = []
    vars = []
    prior_probs = []
    bef_train = time.time()
    classes = np.unique(y_train)
    for c in classes:
        class_samples = X_train[y_train==c]
        means.append(np.mean(class_samples, axis=0))
        vars.append(np.var(class_samples, axis=0) + 1e-6)
        prior_probs.append(probability(y_train, c))
    aft_train = time.time()
    training_time = aft_train-bef_train

    predictions = []
    
    bef_test = time.time()
    classes = np.unique(y_train)
    for sample in X_test:
        posteriors = []
        for i, c in enumerate(classes):
            probs = []
            for feature in range(len(sample)):
                mean = means[i][feature]
                var = vars[i][feature]
                x= sample[feature]
                gaus_prob = (1/np.sqrt(2*np.pi*var)) * np.exp(-((x-mean)**2)/(2*var))
                probs.append(gaus_prob)
            likelihood = np.prod(probs)

            posteriors.append(prior_probs[i] * likelihood)

        predictions.append(classes[np.argmax(posteriors)])
    aft_test = time.time()
    testing_time = aft_test-bef_test
    return predictions, training_time, testing_time

In [ ]:
columns = ["ID", "RI", "NA2O", "MGO", "AL2O3", "SIO2", "K2O", "CAO", "BAO", "FE2O3", "TYPE", "CAMG"]
feature_cols = ["RI", "NA2O", "MGO", "AL2O3", "SIO2", "K2O", "CAO", "BAO", "FE2O3"]
glass_df = pd.read_csv("glass+identification/glass.data", header=None, names=columns)


y = glass_df["TYPE"].to_numpy()


X = glass_df[feature_cols].to_numpy()


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, seed=39590011, testsize=0.2)

scaler = MyStandardScaler()

scaler.fit(X_train)
scaled_X_train = scaler.transform(X_train)
scaled_X_test = scaler.transform(X_test)


In [ ]:
#KNN results
KNN_pred, KNN_training_time, KNN_testing_time = KNNClassifier(X_train, X_test, Y_train, 7)
KNN_accuracy = accuracy_score(Y_test, KNN_pred)
KNN_cm = confusion_matrix(Y_test, KNN_pred)

print(f'KNN training time:{KNN_training_time} seconds')
print(f'KNN testing time:{KNN_testing_time} seconds')
print(f'KNN accuracy score:{KNN_accuracy*100}%')
print(f'KNN confusion matrix:\n{KNN_cm}')

#Decision Tree Results
bef_train = time.time()
tree = build_tree(X_train, Y_train, 0, max_depth=3)
aft_train = time.time()
DT_training_time = aft_train-bef_train
bef_test = time.time()
DT_pred = DTClassifier(tree, X_test)
aft_test = time.time()
DT_testing_time = aft_test-bef_test
DT_accuracy = accuracy_score(Y_test, DT_pred)
DT_cm = confusion_matrix(Y_test, DT_pred)

print(f'DT training time:{DT_training_time} seconds')
print(f'DT testing time:{DT_testing_time} seconds')
print(f'DT accuracy score:{DT_accuracy*100}%')
print(f'DT confusion matrix:\n{DT_cm}')

#Naive Bayes Results
NB_pred, NB_training_time, NB_testing_time = NBClassifier(X_test, X_train, Y_train)
NB_accuracy = accuracy_score(Y_test, NB_pred)
NB_cm = confusion_matrix(Y_test, NB_pred)

print(f'NB training time:{NB_training_time} seconds')
print(f'NB testing time:{NB_testing_time} seconds')
print(f'NB accuracy score:{NB_accuracy*100}%')
print(f'NB confusion matrix:\n{NB_cm}')

KNN training time:0.0007195472717285156 seconds
KNN testing time:0.0004096031188964844 seconds
KNN accuracy score:78.57142857142857%
KNN confusion matrix:
[[12  2  0  0  0  0]
 [ 1 12  1  0  2  0]
 [ 1  0  1  0  0  0]
 [ 0  0  0  1  0  0]
 [ 0  0  0  0  1  0]
 [ 1  1  0  0  0  6]]
DT training time:0.15093517303466797 seconds
DT testing time:9.274482727050781e-05 seconds
DT accuracy score:73.80952380952381%
DT confusion matrix:
[[13  1  0  0  0  0]
 [ 3 12  0  1  0  0]
 [ 1  1  0  0  0  0]
 [ 0  0  0  1  0  0]
 [ 0  0  0  1  0  0]
 [ 1  0  0  2  0  5]]
NB training time:0.0003352165222167969 seconds
NB testing time:0.002509593963623047 seconds
NB accuracy score:38.095238095238095%
NB confusion matrix:
[[ 1  0 13  0  0  0]
 [ 0  4 12  0  0  0]
 [ 1  0  1  0  0  0]
 [ 0  0  0  1  0  0]
 [ 0  0  0  0  1  0]
 [ 0  0  0  0  0  8]]
